# 모델 3. 숙소/현재 위치 기반 순차 관광지 추천 엔진

이 노트북은 기존 모델 1, 모델 2 결과를 기반으로 **API 입력값을 받아 현재 위치 기준 관광지 Top 5를 추천**하는 구조입니다.

- 모델 1 output: 관광지 유형별 확률
- 모델 2 output: 기본 추천 점수
- 모델 3 input: 숙소/현재 위치 좌표, 동반유형, 동반인원, 선호 테마, 날씨, 미세먼지, 방문 완료 목록
- 모델 3 output: 현재 위치 기준 관광지 추천 Top 5

API 구현은 포함하지 않고, API에서 들어올 값을 `request_schema` 형태로 가정합니다.


In [10]:
from pathlib import Path
import math
import json
import unicodedata
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)


## 1. 경로 설정

현재 노트북을 `KTO/src/` 안에서 실행하는 구조를 기본으로 합니다.

아래 경로를 자동으로 탐색합니다.

1. `src/tourism_model2_outputs/model2_recommend_score_dataset.csv`
2. 없으면 `../src/tourism_model2_outputs/model2_recommend_score_dataset.csv`
3. 없으면 `/mnt/data/tourism_model2_outputs/model2_recommend_score_dataset.csv`


In [11]:
def normalize_text(text):
    return unicodedata.normalize("NFC", str(text))


def read_csv_auto(path):
    """UTF-8, UTF-8-SIG, CP949 순서로 CSV를 읽습니다."""
    path = Path(path)
    encodings = ["utf-8-sig", "utf-8", "cp949", "euc-kr"]
    last_error = None
    for enc in encodings:
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception as e:
            last_error = e
    raise last_error


def find_existing_path(candidates):
    for p in candidates:
        p = Path(p)
        if p.exists():
            return p
    raise FileNotFoundError("다음 후보 경로에서 파일을 찾지 못했습니다:" + "".join(map(str, candidates)))

# 현재 노트북이 KTO/src 안에 있을 때를 기준으로 설계
BASE_DIR = Path("..").resolve()

MODEL2_DATA_PATH = find_existing_path([
    Path("recommend_score_model_outputs") / "model2_recommend_score_dataset.csv",
    BASE_DIR / "src" / "recommend_score_model_outputs" / "model2_recommend_score_dataset.csv",
    BASE_DIR / "recommend_score_model_outputs" / "model2_recommend_score_dataset.csv",
    Path("/mnt/data/recommend_score_model_outputs/model2_recommend_score_dataset.csv")
])

OUTPUT_DIR = Path("route_recommendation_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print("MODEL2_DATA_PATH:", MODEL2_DATA_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())


MODEL2_DATA_PATH: recommend_score_model_outputs/model2_recommend_score_dataset.csv
OUTPUT_DIR: /Users/jy-kim/Documents/GitHub/KTO/src/route_recommendation_outputs


## 2. 모델 2 결과 데이터 불러오기

모델 3은 모델 2에서 생성한 `model2_recommend_score_dataset.csv`를 기본 데이터로 사용합니다.


In [12]:
df = read_csv_auto(MODEL2_DATA_PATH)
print(df.shape)
df.head()


(381, 36)


,contentid,recommend_rank,title,addr1,content_type_name,target_tourism_type,type_top1,type_top1_prob,recommend_score,predicted_recommend_score,nav_score,type_quality_score,info_score,recent_score,regional_demand_score,type_prob_관광지,type_prob_레저스포츠,type_prob_문화관광,type_prob_쇼핑,type_prob_숙박,type_prob_역사관광,type_prob_음식,type_prob_자연관광,type_prob_체험관광,avg_monthly_visitors,avg_visitor_growth_rate,avg_stay_time,avg_overnight_days,avg_overnight_ratio,avg_tourism_spending_outsider,avg_local_currency_spending,mapx,mapy,has_image,has_tel,has_addr
0,3013584,1,카페 뷰66 미호점,경기도 남양주시 강변북로632번길 66 (수석동),음식점,음식,음식,0.839223,72.799972,72.154945,1.0,0.124004,0.9,0.998155,0.583333,0.030885,0.027046,0.016593,0.021864,0.014909,0.024516,0.839223,0.019948,0.005016,5.961420e+06,2.491667,1239.0,3.604167,7.275,0,11009367.0,127.176092,37.587323,1,0,1
1,3013767,2,카페스토리,경기도 남양주시 와부읍 팔당로81번길 74-65,음식점,음식,음식,0.844068,72.692983,72.145216,1.0,0.129781,0.9,0.968635,0.583333,0.036605,0.021513,0.022371,0.020043,0.006109,0.020669,0.844068,0.021765,0.006856,5.961420e+06,2.491667,1239.0,3.604167,7.275,0,11009367.0,127.241313,37.554867,1,0,1
2,798518,3,가든갤러리,경기도 남양주시 강변북로632번길 57-28 (수석동),음식점,음식,음식,0.763177,72.524588,71.926292,1.0,0.192177,0.9,0.948339,0.583333,0.035424,0.048079,0.038129,0.031096,0.013550,0.030632,0.763177,0.033056,0.006856,5.961420e+06,2.491667,1239.0,3.604167,7.275,0,11009367.0,127.174134,37.587745,1,0,1
3,2863065,4,카펜트리,경기도 남양주시 진접읍 경복대로212번길 15,음식점,음식,음식,0.840827,71.967119,71.737164,1.0,0.115291,0.9,0.929889,0.583333,0.027038,0.018573,0.016055,0.028783,0.015099,0.026707,0.840827,0.018220,0.008698,5.961420e+06,2.491667,1239.0,3.604167,7.275,0,11009367.0,127.205596,37.717002,1,0,1
4,799603,5,초대,경기도 남양주시 강변북로632번길 6-45,음식점,음식,음식,0.816695,71.608709,71.735155,1.0,0.153292,0.9,0.854244,0.583333,0.031430,0.031597,0.030879,0.021138,0.008875,0.024860,0.816695,0.027671,0.006856,5.961420e+06,2.491667,1239.0,3.604167,7.275,0,11009367.0,127.171547,37.585545,1,0,1


## 3. 필수 컬럼 확인 및 기본 전처리

모델 3에서 최소한 필요한 컬럼은 다음과 같습니다.

- 관광지 식별: `contentid`, `title`
- 위치: `mapx`, `mapy`
- 기본 추천 점수: `predicted_recommend_score` 또는 `recommend_score`
- 관광지 유형 확률: `type_prob_...`


In [13]:
# 추천 점수 컬럼 자동 선택
if "predicted_recommend_score" in df.columns:
    BASE_SCORE_COL = "predicted_recommend_score"
elif "recommend_score" in df.columns:
    BASE_SCORE_COL = "recommend_score"
else:
    raise KeyError("predicted_recommend_score 또는 recommend_score 컬럼이 필요합니다.")

required_cols = ["title", "mapx", "mapy", BASE_SCORE_COL]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise KeyError(f"필수 컬럼이 없습니다: {missing}")

# contentid가 없으면 index 기반 ID 생성
if "contentid" not in df.columns:
    df["contentid"] = np.arange(len(df))

# 좌표/점수 숫자형 변환
df["mapx"] = pd.to_numeric(df["mapx"], errors="coerce")
df["mapy"] = pd.to_numeric(df["mapy"], errors="coerce")
df[BASE_SCORE_COL] = pd.to_numeric(df[BASE_SCORE_COL], errors="coerce")

# 좌표 없는 관광지는 추천 후보에서 제외
df = df.dropna(subset=["mapx", "mapy"]).copy()

# 유형 확률 컬럼 확인
type_prob_cols = [c for c in df.columns if c.startswith("type_prob_")]
print("기본 추천 점수 컬럼:", BASE_SCORE_COL)
print("유형 확률 컬럼 수:", len(type_prob_cols))
print(type_prob_cols)


기본 추천 점수 컬럼: predicted_recommend_score
유형 확률 컬럼 수: 9
['type_prob_관광지', 'type_prob_레저스포츠', 'type_prob_문화관광', 'type_prob_쇼핑', 'type_prob_숙박', 'type_prob_역사관광', 'type_prob_음식', 'type_prob_자연관광', 'type_prob_체험관광']


## 4. API 입력값 스키마 예시

실제 서비스에서는 아래 값들이 API 요청으로 들어오면 됩니다. 지금은 테스트용 예시값으로 둡니다.

`current_location`은 처음에는 숙소 좌표이고, 다음 추천에서는 사용자가 선택한 관광지 좌표가 됩니다.


In [25]:
request_schema = {
    "current_location": {
        "name": "경복대학교 남양주 캠퍼스",
        "mapx": 127.211247,
        "mapy": 37.734991
    },
    "companion": {
        "type": "혼자",        # 혼자, 연인, 친구, 가족, 아이동반, 부모님, 단체
        "count": 1
    },
    "preference": {
        "themes": ["레저스포츠", "음식"]  # 자연관광, 역사관광, 문화관광, 레저스포츠, 음식, 쇼핑, 숙박, 관광지
    },
    "environment": {
        "weather": "흐림",     # 맑음, 흐림, 비, 눈, 폭염, 한파
        "fine_dust": "나쁨"    # 좋음, 보통, 나쁨, 매우나쁨
    },
    "route": {
        "visited_contentids": [],
        "top_n": 5,
        "max_distance_km": 10  # 예: 20. None이면 거리 제한 없음
    }
}

request_schema


{'current_location': {'name': '경복대학교 남양주 캠퍼스',
  'mapx': 127.211247,
  'mapy': 37.734991},
 'companion': {'type': '혼자', 'count': 1},
 'preference': {'themes': ['레저스포츠', '음식']},
 'environment': {'weather': '흐림', 'fine_dust': '나쁨'},
 'route': {'visited_contentids': [], 'top_n': 5, 'max_distance_km': 10}}

## 5. 추천 점수 계산 함수 정의

최종 점수는 다음 항목을 결합합니다.

```text
final_score =
기본 추천 점수 × 0.30
+ 테마 적합도 × 0.20
+ 동반 유형 적합도 × 0.15
+ 날씨 적합도 × 0.15
+ 미세먼지 적합도 × 0.08
+ 거리 점수 × 0.09
+ 동반 인원 점수 × 0.03
```


In [15]:
def minmax(series):
    s = pd.to_numeric(series, errors="coerce").fillna(0)
    min_v = s.min()
    max_v = s.max()
    if max_v == min_v:
        return pd.Series(np.ones(len(s)), index=s.index)
    return (s - min_v) / (max_v - min_v)


def haversine_km(lon1, lat1, lon2, lat2):
    """경도/위도 좌표 간 거리(km)를 계산합니다."""
    R = 6371.0
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c


def get_type_col(theme):
    """테마명을 type_prob 컬럼명으로 변환합니다."""
    theme = str(theme).strip().replace("/", "_").replace(" ", "_")
    return f"type_prob_{theme}"


def safe_col(df, col, default=0.0):
    if col in df.columns:
        return pd.to_numeric(df[col], errors="coerce").fillna(default)
    return pd.Series(default, index=df.index)


In [16]:
def compute_theme_match_score(data, themes):
    """선호 테마와 관광지 유형 확률의 일치도를 계산합니다."""
    if not themes:
        return pd.Series(0.5, index=data.index)

    score = pd.Series(0.0, index=data.index)
    valid_count = 0

    alias = {
        "자연": "자연관광",
        "역사": "역사관광",
        "역사문화": "역사관광",
        "문화": "문화관광",
        "체험": "문화관광",
        "레저": "레저스포츠",
        "스포츠": "레저스포츠",
        "맛집": "음식",
        "카페": "음식",
        "식당": "음식",
        "숙소": "숙박",
    }

    for theme in themes:
        mapped_theme = alias.get(str(theme), str(theme))
        col = get_type_col(mapped_theme)
        if col in data.columns:
            score += safe_col(data, col)
            valid_count += 1

    if valid_count == 0:
        return pd.Series(0.5, index=data.index)

    return score / valid_count


# 여기 아래 부분의 동반 유형별 적합도는 데이터 분석 후 수정

In [17]:
def weighted_type_score(data, weights):
    score = pd.Series(0.0, index=data.index)
    for col, w in weights.items():
        score += safe_col(data, col) * w
    return score


def compute_companion_score(data, companion_type):
    """동반 유형별 적합도를 계산합니다."""
    companion_type = str(companion_type).strip()
    weights = {
        "혼자": {"type_prob_자연관광": 0.35, "type_prob_문화관광": 0.30, "type_prob_음식": 0.20, "type_prob_관광지": 0.15},
        "연인": {"type_prob_자연관광": 0.35, "type_prob_음식": 0.30, "type_prob_문화관광": 0.20, "type_prob_관광지": 0.15},
        "친구": {"type_prob_레저스포츠": 0.35, "type_prob_음식": 0.30, "type_prob_관광지": 0.20, "type_prob_문화관광": 0.15},
        "가족": {"type_prob_자연관광": 0.30, "type_prob_역사관광": 0.25, "type_prob_문화관광": 0.25, "type_prob_관광지": 0.20},
        "아이동반": {"type_prob_문화관광": 0.30, "type_prob_관광지": 0.30, "type_prob_자연관광": 0.25, "type_prob_음식": 0.15},
        "부모님": {"type_prob_역사관광": 0.35, "type_prob_자연관광": 0.30, "type_prob_문화관광": 0.25, "type_prob_음식": 0.10},
        "단체": {"type_prob_관광지": 0.30, "type_prob_음식": 0.25, "type_prob_레저스포츠": 0.25, "type_prob_문화관광": 0.20},
    }
    return weighted_type_score(data, weights.get(companion_type, weights["가족"]))


def compute_weather_score(data, weather):
    """날씨별 적합도를 계산합니다."""
    weather = str(weather).strip()
    weights = {
        "맑음": {"type_prob_자연관광": 0.35, "type_prob_레저스포츠": 0.25, "type_prob_관광지": 0.25, "type_prob_음식": 0.15},
        "흐림": {"type_prob_문화관광": 0.30, "type_prob_역사관광": 0.25, "type_prob_음식": 0.25, "type_prob_관광지": 0.20},
        "비": {"type_prob_음식": 0.30, "type_prob_쇼핑": 0.25, "type_prob_문화관광": 0.25, "type_prob_숙박": 0.20},
        "눈": {"type_prob_음식": 0.30, "type_prob_쇼핑": 0.25, "type_prob_문화관광": 0.25, "type_prob_역사관광": 0.20},
        "폭염": {"type_prob_음식": 0.30, "type_prob_쇼핑": 0.25, "type_prob_문화관광": 0.25, "type_prob_숙박": 0.20},
        "한파": {"type_prob_음식": 0.30, "type_prob_쇼핑": 0.25, "type_prob_문화관광": 0.25, "type_prob_숙박": 0.20},
    }
    return weighted_type_score(data, weights.get(weather, weights["흐림"]))


def compute_fine_dust_score(data, fine_dust):
    """미세먼지 상태별 적합도를 계산합니다."""
    fine_dust = str(fine_dust).strip()
    weights = {
        "좋음": {"type_prob_자연관광": 0.35, "type_prob_레저스포츠": 0.25, "type_prob_관광지": 0.25, "type_prob_음식": 0.15},
        "보통": {"type_prob_관광지": 0.25, "type_prob_자연관광": 0.25, "type_prob_문화관광": 0.20, "type_prob_음식": 0.20, "type_prob_역사관광": 0.10},
        "나쁨": {"type_prob_음식": 0.30, "type_prob_쇼핑": 0.25, "type_prob_문화관광": 0.25, "type_prob_숙박": 0.20},
        "매우나쁨": {"type_prob_음식": 0.30, "type_prob_쇼핑": 0.30, "type_prob_문화관광": 0.25, "type_prob_숙박": 0.15},
    }
    return weighted_type_score(data, weights.get(fine_dust, weights["보통"]))


In [18]:
def compute_group_size_score(data, companion_count):
    """동반 인원에 따른 약한 보정 점수입니다."""
    try:
        count = int(companion_count)
    except Exception:
        count = 2

    if count <= 1:
        weights = {"type_prob_음식": 0.35, "type_prob_문화관광": 0.30, "type_prob_자연관광": 0.20, "type_prob_관광지": 0.15}
    elif count == 2:
        weights = {"type_prob_자연관광": 0.30, "type_prob_음식": 0.30, "type_prob_문화관광": 0.25, "type_prob_관광지": 0.15}
    elif count <= 4:
        weights = {"type_prob_관광지": 0.30, "type_prob_자연관광": 0.25, "type_prob_음식": 0.25, "type_prob_레저스포츠": 0.20}
    else:
        weights = {"type_prob_관광지": 0.35, "type_prob_음식": 0.30, "type_prob_레저스포츠": 0.20, "type_prob_숙박": 0.15}
    return weighted_type_score(data, weights)


## 6. 현재 위치 기준 Top 5 추천 함수

이 함수가 실제 API 엔드포인트에서 호출될 핵심 로직입니다.


In [19]:
def make_recommend_reason(row, themes, companion_type, weather, fine_dust):
    reasons = []
    if themes:
        reasons.append(f"선호 테마({', '.join(map(str, themes))}) 적합도 {row.get('theme_match_score', 0):.2f}")
    reasons.append(f"동반유형({companion_type}) 적합도 {row.get('companion_score', 0):.2f}")
    reasons.append(f"날씨({weather}) 적합도 {row.get('weather_score', 0):.2f}")
    reasons.append(f"미세먼지({fine_dust}) 적합도 {row.get('fine_dust_score', 0):.2f}")
    reasons.append(f"현재 위치에서 {row.get('current_distance_km', 0):.1f}km")
    return " / ".join(reasons)


def recommend_next_places(base_df, request, top_n=None, save_path=None):
    """현재 위치와 사용자 조건을 기반으로 다음 관광지 Top-N을 추천합니다."""
    data = base_df.copy()

    current = request.get("current_location", {})
    companion = request.get("companion", {})
    preference = request.get("preference", {})
    environment = request.get("environment", {})
    route = request.get("route", {})

    current_mapx = float(current.get("mapx"))
    current_mapy = float(current.get("mapy"))
    companion_type = companion.get("type", "가족")
    companion_count = companion.get("count", 2)
    themes = preference.get("themes", [])
    weather = environment.get("weather", "흐림")
    fine_dust = environment.get("fine_dust", "보통")
    visited_contentids = set(map(str, route.get("visited_contentids", [])))
    max_distance_km = route.get("max_distance_km", None)
    top_n = int(top_n or route.get("top_n", 5))

    # 방문 완료 관광지 제외
    data["contentid_str"] = data["contentid"].astype(str)
    data["visited_flag"] = data["contentid_str"].isin(visited_contentids).astype(int)
    data = data[data["visited_flag"] == 0].copy()

    # 현재 위치 기준 거리 계산
    data["current_mapx"] = current_mapx
    data["current_mapy"] = current_mapy
    data["current_distance_km"] = haversine_km(
        current_mapx, current_mapy,
        data["mapx"].astype(float), data["mapy"].astype(float)
    )

    if max_distance_km is not None:
        data = data[data["current_distance_km"] <= float(max_distance_km)].copy()

    # 점수 계산
    data["base_score_norm"] = minmax(data[BASE_SCORE_COL])
    data["distance_score"] = 1 / (1 + data["current_distance_km"])
    data["theme_match_score"] = compute_theme_match_score(data, themes)
    data["companion_score"] = compute_companion_score(data, companion_type)
    data["weather_score"] = compute_weather_score(data, weather)
    data["fine_dust_score"] = compute_fine_dust_score(data, fine_dust)
    data["group_size_score"] = compute_group_size_score(data, companion_count)

    data["final_score"] = (
        data["base_score_norm"] * 0.30 +
        data["theme_match_score"] * 0.20 +
        data["companion_score"] * 0.15 +
        data["weather_score"] * 0.15 +
        data["fine_dust_score"] * 0.08 +
        data["distance_score"] * 0.09 +
        data["group_size_score"] * 0.03
    )
    data["final_score_100"] = data["final_score"] * 100
    data["recommend_reason"] = data.apply(lambda row: make_recommend_reason(row, themes, companion_type, weather, fine_dust), axis=1)

    result = data.sort_values(["final_score", "current_distance_km"], ascending=[False, True]).head(top_n).copy()

    output_cols = [
        "contentid", "title", "addr1", "content_type_name", "type_top1", "type_top1_prob",
        "mapx", "mapy", "current_distance_km", BASE_SCORE_COL,
        "base_score_norm", "theme_match_score", "companion_score", "weather_score",
        "fine_dust_score", "distance_score", "group_size_score", "final_score_100", "recommend_reason"
    ]
    output_cols = [c for c in output_cols if c in result.columns]
    result = result[output_cols].reset_index(drop=True)
    result.insert(0, "rank", np.arange(1, len(result) + 1))

    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        result.to_csv(save_path, index=False, encoding="utf-8-sig")

    return result


## 7. 1차 추천: 숙소 또는 현재 위치 기준 Top 5

API에서 숙소 좌표가 들어온 상황을 가정하고 추천을 실행합니다.


In [26]:
top5_from_accommodation = recommend_next_places(
    df,
    request_schema,
    save_path=OUTPUT_DIR / "step1_top5_from_current_location.csv"
)

top5_from_accommodation


,rank,contentid,title,addr1,content_type_name,type_top1,type_top1_prob,mapx,mapy,current_distance_km,predicted_recommend_score,base_score_norm,theme_match_score,companion_score,weather_score,fine_dust_score,distance_score,group_size_score,final_score_100,recommend_reason
0,1,2863065,카펜트리,경기도 남양주시 진접읍 경복대로212번길 15,음식점,음식,0.840827,127.205596,37.717002,2.061094,71.737164,1.000000,0.429700,0.183415,0.227108,0.266477,0.326681,0.306806,50.744198,"선호 테마(레저스포츠, 음식) 적합도 0.43 / 동반유형(혼자) 적합도 0.18 ..."
1,2,615484,광릉돌솥밥,경기도 남양주시 진접읍 금강로1845번길 1,음식점,음식,0.823304,127.208684,37.749647,1.645190,71.207241,0.983487,0.424746,0.181658,0.223582,0.260973,0.378045,0.302226,50.475000,"선호 테마(레저스포츠, 음식) 적합도 0.42 / 동반유형(혼자) 적합도 0.18 ..."
2,3,608010,사랑방,경기도 남양주시 진접읍 광릉수목원로 187-1,음식점,음식,0.871311,127.187986,37.747411,2.467956,70.234322,0.953170,0.450283,0.185341,0.227629,0.268817,0.288354,0.314006,49.483033,"선호 테마(레저스포츠, 음식) 적합도 0.45 / 동반유형(혼자) 적합도 0.19 ..."
3,4,2876677,수사골,경기도 남양주시 오남읍 진건오남로708번길 53,음식점,음식,0.819255,127.214134,37.698399,4.076727,71.492115,0.992364,0.423000,0.181732,0.223853,0.260072,0.196977,0.301722,49.073231,"선호 테마(레저스포츠, 음식) 적합도 0.42 / 동반유형(혼자) 적합도 0.18 ..."
4,5,2876761,돼지마을순대국,경기도 남양주시 진접읍 진벌로 57,음식점,음식,0.992559,127.211797,37.745090,1.124053,65.924100,0.818859,0.496909,0.199004,0.248364,0.298955,0.470798,0.347769,48.886601,"선호 테마(레저스포츠, 음식) 적합도 0.50 / 동반유형(혼자) 적합도 0.20 ..."


## 8. 선택된 관광지를 다음 현재 위치로 설정

사용자가 추천 결과 중 1개를 선택하면, 해당 관광지를 새로운 현재 위치로 바꿉니다.

아래 예시는 1순위 관광지를 선택했다고 가정합니다.


In [27]:
selected_place = top5_from_accommodation.iloc[0]

next_request = json.loads(json.dumps(request_schema, ensure_ascii=False))
next_request["current_location"] = {
    "name": selected_place["title"],
    "mapx": float(selected_place["mapx"]),
    "mapy": float(selected_place["mapy"])
}

visited = set(map(str, next_request["route"].get("visited_contentids", [])))
visited.add(str(selected_place["contentid"]))
next_request["route"]["visited_contentids"] = list(visited)

next_request


{'current_location': {'name': '카펜트리',
  'mapx': 127.205595705715,
  'mapy': 37.7170021565706},
 'companion': {'type': '혼자', 'count': 1},
 'preference': {'themes': ['레저스포츠', '음식']},
 'environment': {'weather': '흐림', 'fine_dust': '나쁨'},
 'route': {'visited_contentids': ['2863065'],
  'top_n': 5,
  'max_distance_km': 10}}

## 9. 2차 추천: 선택 관광지 기준 다음 Top 5

현재 위치가 숙소에서 선택 관광지로 바뀐 후, 다시 Top 5를 추천합니다.


In [28]:
top5_from_selected_place = recommend_next_places(
    df,
    next_request,
    save_path=OUTPUT_DIR / "step2_top5_from_selected_place.csv"
)

top5_from_selected_place


,rank,contentid,title,addr1,content_type_name,type_top1,type_top1_prob,mapx,mapy,current_distance_km,predicted_recommend_score,base_score_norm,theme_match_score,companion_score,weather_score,fine_dust_score,distance_score,group_size_score,final_score_100,recommend_reason
0,1,2842976,스시쿠모,경기도 남양주시 진접읍 해밀예당1로200번길 13-15,음식점,음식,0.971765,127.201430,37.718627,0.408544,65.816702,0.821787,0.487936,0.196784,0.245063,0.296292,0.709953,0.342378,50.827097,"선호 테마(레저스포츠, 음식) 적합도 0.49 / 동반유형(혼자) 적합도 0.20 ..."
1,2,2876677,수사골,경기도 남양주시 오남읍 진건오남로708번길 53,음식점,음식,0.819255,127.214134,37.698399,2.200684,71.492115,1.000000,0.423000,0.181732,0.223853,0.260072,0.312433,0.301722,50.341414,"선호 테마(레저스포츠, 음식) 적합도 0.42 / 동반유형(혼자) 적합도 0.18 ..."
2,3,615484,광릉돌솥밥,경기도 남양주시 진접읍 금강로1845번길 1,음식점,음식,0.823304,127.208684,37.749647,3.640088,71.207241,0.991055,0.424746,0.181658,0.223582,0.260973,0.215513,0.302226,49.239247,"선호 테마(레저스포츠, 음식) 적합도 0.42 / 동반유형(혼자) 적합도 0.18 ..."
3,4,608010,사랑방,경기도 남양주시 진접읍 광릉수목원로 187-1,음식점,음식,0.871311,127.187986,37.747411,3.719048,70.234322,0.960504,0.450283,0.185341,0.227629,0.268817,0.211907,0.314006,49.015041,"선호 테마(레저스포츠, 음식) 적합도 0.45 / 동반유형(혼자) 적합도 0.19 ..."
4,5,2940879,더에이,경기도 남양주시 비룡로 1438-67,음식점,음식,0.982218,127.277855,37.747891,7.223461,68.828823,0.916370,0.491738,0.197525,0.246368,0.298163,0.121603,0.344740,48.498242,"선호 테마(레저스포츠, 음식) 적합도 0.49 / 동반유형(혼자) 적합도 0.20 ..."


## 10. 여러 단계 코스 자동 생성 함수

사용자가 매 단계에서 직접 선택하는 방식이 기본이지만, 테스트용으로 1순위를 자동 선택하면서 코스를 생성하는 함수도 제공합니다.


In [29]:
def generate_route_by_top1(base_df, request, steps=3):
    route_request = json.loads(json.dumps(request, ensure_ascii=False))
    route_rows = []

    for step in range(1, steps + 1):
        recs = recommend_next_places(base_df, route_request, top_n=5)
        if recs.empty:
            break

        selected = recs.iloc[0].copy()
        selected["step"] = step
        selected["start_name"] = route_request["current_location"].get("name", "현재 위치")
        route_rows.append(selected)

        visited = set(map(str, route_request["route"].get("visited_contentids", [])))
        visited.add(str(selected["contentid"]))
        route_request["route"]["visited_contentids"] = list(visited)
        route_request["current_location"] = {
            "name": selected["title"],
            "mapx": float(selected["mapx"]),
            "mapy": float(selected["mapy"])
        }

    if len(route_rows) == 0:
        return pd.DataFrame()

    route_df = pd.DataFrame(route_rows)
    front_cols = ["step", "start_name", "rank", "contentid", "title", "current_distance_km", "final_score_100", "recommend_reason"]
    front_cols = [c for c in front_cols if c in route_df.columns]
    route_df = route_df[front_cols + [c for c in route_df.columns if c not in front_cols]]
    return route_df.reset_index(drop=True)


auto_route = generate_route_by_top1(df, request_schema, steps=3)
auto_route.to_csv(OUTPUT_DIR / "auto_route_top1_sequence.csv", index=False, encoding="utf-8-sig")
auto_route


,step,start_name,rank,contentid,title,current_distance_km,final_score_100,recommend_reason,addr1,content_type_name,type_top1,type_top1_prob,mapx,mapy,predicted_recommend_score,base_score_norm,theme_match_score,companion_score,weather_score,fine_dust_score,distance_score,group_size_score
0,1,경복대학교 남양주 캠퍼스,1,2863065,카펜트리,2.061094,50.744198,"선호 테마(레저스포츠, 음식) 적합도 0.43 / 동반유형(혼자) 적합도 0.18 ...",경기도 남양주시 진접읍 경복대로212번길 15,음식점,음식,0.840827,127.205596,37.717002,71.737164,1.000000,0.429700,0.183415,0.227108,0.266477,0.326681,0.306806
1,2,카펜트리,1,2842976,스시쿠모,0.408544,50.827097,"선호 테마(레저스포츠, 음식) 적합도 0.49 / 동반유형(혼자) 적합도 0.20 ...",경기도 남양주시 진접읍 해밀예당1로200번길 13-15,음식점,음식,0.971765,127.201430,37.718627,65.816702,0.821787,0.487936,0.196784,0.245063,0.296292,0.709953,0.342378
2,3,스시쿠모,1,2876677,수사골,2.511531,50.092500,"선호 테마(레저스포츠, 음식) 적합도 0.42 / 동반유형(혼자) 적합도 0.18 ...",경기도 남양주시 오남읍 진건오남로708번길 53,음식점,음식,0.819255,127.214134,37.698399,71.492115,1.000000,0.423000,0.181732,0.223853,0.260072,0.284776,0.301722


## 11. API 연동 시 사용할 입력/출력 정리

### API 입력값

```json
{
  "current_location": {"name": "숙소명", "mapx": 127.2165, "mapy": 37.6360},
  "companion": {"type": "연인", "count": 2},
  "preference": {"themes": ["자연관광", "음식"]},
  "environment": {"weather": "맑음", "fine_dust": "보통"},
  "route": {"visited_contentids": [], "top_n": 5, "max_distance_km": null}
}
```

### API 출력값

- 관광지명
- 대표 유형
- 거리
- 추천 점수
- 추천 이유
- 좌표
- contentid

### 다음 단계 추천

사용자가 특정 관광지를 선택하면 해당 관광지의 `mapx`, `mapy`, `contentid`를 사용해 다음 요청을 구성하면 됩니다.


In [30]:
print("생성된 결과 파일")
for p in sorted(OUTPUT_DIR.glob("*.csv")):
    print("-", p)


생성된 결과 파일
- route_recommendation_outputs/auto_route_top1_sequence.csv
- route_recommendation_outputs/step1_top5_from_current_location.csv
- route_recommendation_outputs/step2_top5_from_selected_place.csv
